## Time to get your hands dirty. Your first neural network; pick your favourite.

*(exam guidelines available [here](https://github.com/dgerosa/astrostatistics_bicocca_2026/blob/main/exams.md))*


For the last coding assignment, you'll need to implement a neural network. We'll look at a relatively simple binary classification problem. Here below are three options; completing one of them for the exam is enough

### Tasks:

1. Remember: scale your data appropriately

2. Decide on a testing strategy (a simple test/train split? a CV strategy? set a test set aside to be looked at at the very end?)

2. Decide your optimization metric.

3. Write down your network architecture. You can start from a fully connected, multi-layer perceptron (and then explore)

4. Use one the package among those we've seen. These include Tensorflow via keras, pytorch, and the MPL classifier implemented in scikit-learn. This is an opportunity to pick the one you're most interested in learning. 

5. Optimize the hyperparameters of your network. Explore different hyperparameters and see what fits the data best.  Do your best now to optimize the network architecture. Be creative!

6. Report on the perfomance of the network on the test set; report other metrics that have not been optimized.


### A few tips:

- In scikit-learn, remember that you can utilize all availables cores on your machine with `n_jobs=-1`. Print out the classification score for the training data, and the best parameters obtained by the cross validation.
- If it takes too long, run the hyperparameter optimization on a subset of the training set. Then retrain the full network using the best hyperparameters only.
- On cross validation, for scikit learn we've seen how to use `GridSearchCV` already. For Tensorflow, there's a really cool tool called [Tensorboard](https://www.tensorflow.org/tensorboard)

### Datasets:

You can choose one of these three problems:

- **1. Galaxies vs quasars (but with neural networks)** Go back to our SDSS data we've used in Lecture 19. We had color differences, and the task was to classifty quasars vs galaxies. Repeat that task with a neural network.

- **2. Can a computer learn if we're going to detect gravitational waves? (but with neural networks)** Go back to the SNR classifier for gravitational wave events, same data we've used in Lecture. We had properties of black hole binaries, and the task was to classify. Repeat that task with a neural network.

- **3. The HiggsML challenge** Branching out of astrophysics, let's mess around with a dataset of simulated but  realistic events from the ATLAS particle detector at CERN.
    - Data are at `solutions/higgs.tar.gz` (you need to uncompress with `tar -czvf`)
    - There are $N_{\rm samples} = 2.5\times 10^5$ entries with $N_{\rm features}=30$ features each. 
    - The taks is that of classifying these features against a set of labels, which are either `s` (source) or `b` (background).
    - For some info on both the physics and the dataset see [this document](https://higgsml.lal.in2p3.fr/files/2014/04/documentation_v1.8.pdf); includes a description of the features and how data have been padded (-999) for missing values.
    - This dataset was part of a challenge that run on Kaggle in 2014: https://higgsml.ijclab.in2p3.fr/ 




In [1]:
import pandas as pd 
import tensorflow
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

I0000 00:00:1782554216.449553   16233 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782554216.450068   16233 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1782554216.506952   16233 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1782554215.628278   16233 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

In [2]:
df = pd.read_csv('/home/matti/uni/astrostatistics_bicocca_2026/solutions/higgs.csv')


In [3]:
df_filt = df.loc[:, ~df.columns.isin(['EventId','Label', 'KaggleSet', 'KaggleWeight', 'Weight'])]

In [4]:
y = df['Label']

In [5]:
y = np.array([1 if label == 's' else 0 for label in y], dtype='int32')

In [6]:
y

array([1, 0, 0, ..., 1, 0, 0], shape=(250000,), dtype=int32)

In [7]:
df_filt

,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,DER_sum_pt,...,PRI_met_phi,PRI_met_sumet,PRI_jet_num,PRI_jet_leading_pt,PRI_jet_leading_eta,PRI_jet_leading_phi,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_subleading_phi,PRI_jet_all_pt
0,138.470,51.655,97.827,27.980,0.91,124.711,2.666,3.064,41.928,197.760,...,-0.277,258.733,2,67.435,2.150,0.444,46.062,1.24,-2.475,113.497
1,160.937,68.768,103.235,48.146,-999.00,-999.000,-999.000,3.473,2.078,125.157,...,-1.916,164.546,1,46.226,0.725,1.158,-999.000,-999.00,-999.000,46.226
2,-999.000,162.172,125.953,35.635,-999.00,-999.000,-999.000,3.148,9.336,197.814,...,-2.186,260.414,1,44.251,2.053,-2.028,-999.000,-999.00,-999.000,44.251
3,143.905,81.417,80.943,0.414,-999.00,-999.000,-999.000,3.310,0.414,75.968,...,0.060,86.062,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,-0.000
4,175.864,16.915,134.805,16.405,-999.00,-999.000,-999.000,3.891,16.405,57.983,...,-0.871,53.131,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,0.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249995,-999.000,71.989,36.548,5.042,-999.00,-999.000,-999.000,1.392,5.042,55.892,...,2.859,144.665,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,0.000
249996,-999.000,58.179,68.083,22.439,-999.00,-999.000,-999.000,2.585,22.439,50.618,...,-0.867,80.408,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,-0.000
249997,105.457,60.526,75.839,39.757,-999.00,-999.000,-999.000,2.390,22.183,120.462,...,-2.890,198.907,1,41.992,1.800,-0.166,-999.000,-999.00,-999.000,41.992
249998,94.951,19.362,68.812,13.504,-999.00,-999.000,-999.000,3.365,13.504,55.859,...,0.811,112.718,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,0.000


In [8]:
df_filt_nan = df_filt.replace(-999, np.nan)


In [9]:
def rescale(d_train, d_test):
    imputer = SimpleImputer(strategy = 'median', add_indicator= True)
    scaler = StandardScaler()
    d_train_i = imputer.fit_transform(d_train)
    d_train_si = scaler.fit_transform(d_train_i)
    d_test_i = imputer.transform(d_test)
    d_test_si = scaler.transform(d_test_i)
    return d_train_si, d_test_si

I try different models with different number of layers/fuctions/neurons in a cv scheme

In [10]:
from sklearn.model_selection import train_test_split, KFold
from keras import layers

In [11]:
def build(m):
    m.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return m

In [12]:
# keras.backend.clear_session()
def model_sel(select):
    if (select == 0):
    #model1 = 2 layer, all relu with differrent dropouts to prevent overfitting
        model = keras.Sequential([
            layers.Input(shape = (41,)),
            layers.Dense(32, activation='relu', ),
            layers.BatchNormalization(),
            layers.Dropout(0.5),
            layers.Dense(32, activation='relu', ),
            layers.BatchNormalization(),
            layers.Dropout(0.3),
            layers.Dense(1, activation='sigmoid')
        ])
        # model1.summary()
    
    elif (select == 1):
        model = keras.Sequential([
            layers.Input(shape = (41, )),
            layers.Dense(64, activation='relu'),
            layers.Dropout(0.3),
            layers.Dense(128, activation='relu'),
            layers.Dropout(0.3),
            layers.Dense(64, activation='relu'),
            layers.Dropout(0.3),
            layers.Dense(32, activation='relu'),
            layers.Dropout(0.2),
            layers.Dense(1, activation='sigmoid')
        ])
        # model2.summary()
    elif (select == 2):
        model = keras.Sequential([
            layers.Input(shape = (41,)),
            layers.Dense(64, activation='tanh'),
            layers.Dropout(0.3),
            layers.Dense(64, activation='tanh'),
            layers.Dropout(0.2),
            layers.Dense(1, activation='sigmoid')
        ])
        # model3.summary()
    elif (select == 3):
        model = keras.Sequential([
            layers.Input(shape = (41, )),
            layers.Dense(64, activation='tanh'),
            layers.Dropout(0.3),
            layers.Dense(128, activation='tanh'),
            layers.Dropout(0.3),
            layers.Dense(64, activation='tanh'),
            layers.Dropout(0.3),
            layers.Dense(32, activation='tanh'),
            layers.Dropout(0.2),
            layers.Dense(1, activation='sigmoid')
        ])
        # model4.summary()
    elif (select == 4):
    #equal to model 1 but without dropout and batchnormalization
        model = keras.Sequential([
            layers.Input(shape = (41,)),
            layers.Dense(32, activation='relu', ),
            layers.Dense(32, activation='relu', ),
            layers.Dense(1, activation='sigmoid')
        ])
    else: print('error!!!!! --> choose a model')
    return build(model)

In [13]:
data_train, data_test, y_train, y_test = train_test_split(df_filt_nan, y, test_size= 0.15, shuffle= True)
# prova1, prova2, y1, y2 = train_test_split(data_train, y_train, test_size= 0.2)

In [14]:
data_train

,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,DER_sum_pt,...,PRI_met_phi,PRI_met_sumet,PRI_jet_num,PRI_jet_leading_pt,PRI_jet_leading_eta,PRI_jet_leading_phi,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_subleading_phi,PRI_jet_all_pt
198920,NaN,109.155,55.364,42.762,NaN,NaN,NaN,1.766,4.092,108.600,...,2.513,195.581,1,39.976,-2.829,-1.599,NaN,NaN,NaN,39.976
92065,120.487,72.857,75.819,53.435,5.751,911.197,-8.168,2.948,48.663,173.261,...,-0.004,141.705,2,53.664,3.192,2.824,49.020,-2.559,0.640,102.684
247759,142.485,3.426,44.227,277.780,1.127,146.580,-0.317,1.034,5.458,497.729,...,-1.830,565.829,3,225.507,-0.586,0.968,59.730,0.541,1.362,395.247
33659,115.338,28.392,81.591,26.768,NaN,NaN,NaN,2.989,26.768,82.709,...,1.128,268.132,0,NaN,NaN,NaN,NaN,NaN,NaN,0.000
226260,23.145,75.967,20.841,20.122,NaN,NaN,NaN,0.648,24.313,107.283,...,-2.399,138.886,1,43.553,1.190,-1.954,NaN,NaN,NaN,43.553
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80992,95.141,9.391,69.103,41.756,NaN,NaN,NaN,2.135,3.562,119.200,...,-1.153,176.632,1,39.997,2.862,-1.661,NaN,NaN,NaN,39.997
167343,79.922,42.427,58.441,32.119,NaN,NaN,NaN,2.520,0.885,90.883,...,1.317,129.367,1,31.240,3.454,-0.862,NaN,NaN,NaN,31.240
111478,131.787,9.304,84.258,123.600,0.080,39.060,1.744,2.289,5.555,224.256,...,1.649,271.527,2,84.963,1.281,-1.751,41.569,1.361,-1.197,126.532
178582,164.471,54.940,130.179,25.509,NaN,NaN,NaN,3.464,25.509,52.854,...,-0.844,207.612,0,NaN,NaN,NaN,NaN,NaN,NaN,0.000


In [15]:
kfold = KFold(shuffle = True, n_splits = 5)

acc = []
acc_std = []
loss = []
loss_std = []
for selec in [0, 1, 2, 3,4]:
    acc_cv = []
    loss_cv = []
    for (train_index, cv_test_index) in kfold.split(data_train):
        keras.backend.clear_session()
        data_cv_train, data_cv_test = data_train.iloc[train_index], data_train.iloc[cv_test_index]
        y_cv_train, y_cv_test = y_train[train_index], y_train[cv_test_index]
        d_cv_train, d_cv_test = rescale(data_cv_train, data_cv_test)

        model = model_sel(selec)
        model.fit(d_cv_train, y_cv_train, epochs = 5)
        
        l, accuracy = model.evaluate(d_cv_test, y_cv_test)
        acc_cv.append(accuracy)
        loss_cv.append(l)
    
    
    print(f'============================Done {model} =================================')

    acc.append(np.mean(acc_cv))
    acc_std.append(np.std(acc_cv))
    loss.append(np.mean(loss_cv))
    loss_std.append(np.std(loss_cv))

E0000 00:00:1782554218.928839   16233 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/5
5313/5313 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - accuracy: 0.7657 - loss: 0.4879
Epoch 2/5
5313/5313 ━━━━━━━━━━━━━━━━━━━━ 14s 3ms/step - accuracy: 0.8067 - loss: 0.4319
Epoch 3/5
5313/5313 ━━━━━━━━━━━━━━━━━━━━ 14s 3ms/step - accuracy: 0.8115 - loss: 0.4221
Epoch 4/5
5313/5313 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - accuracy: 0.8134 - loss: 0.4177
Epoch 5/5
5313/5313 ━━━━━━━━━━━━━━━━━━━━ 13s 2ms/step - accuracy: 0.8139 - loss: 0.4144
1329/1329 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.8277 - loss: 0.3911
Epoch 1/5
5313/5313 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - accuracy: 0.7660 - loss: 0.4889
Epoch 2/5
5313/5313 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - accuracy: 0.8049 - loss: 0.4341
Epoch 3/5
5313/5313 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - accuracy: 0.8104 - loss: 0.4221
Epoch 4/5
5313/5313 ━━━━━━━━━━━━━━━━━━━━ 13s 2ms/step - accuracy: 0.8129 - loss: 0.4186
Epoch 5/5
5313/5313 ━━━━━━━━━━━━━━━━━━━━ 13s 2ms/step - accuracy: 0.8141 - loss: 0.4150
1329/1329 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step 

In [16]:
print(acc, '\n', acc_std)
print(loss, '\n', loss_std)

[np.float64(0.8300094246864319), np.float64(0.8371764659881592), np.float64(0.8354870557785035), np.float64(0.8340000033378601), np.float64(0.8379011750221252)] 
 [np.float64(0.0025548547302039746), np.float64(0.001992822018808903), np.float64(0.0017135685076606624), np.float64(0.0010538570184056594), np.float64(0.002380514779054119)]
[np.float64(0.38458094000816345), np.float64(0.37018396854400637), np.float64(0.3680279552936554), np.float64(0.3743964672088623), np.float64(0.3622880458831787)] 
 [np.float64(0.004532135965079308), np.float64(0.004868606899087824), np.float64(0.0030496855843870673), np.float64(0.0016031708324422743), np.float64(0.0038186267214412786)]
